# 🧙 O Mago Mestre — API Server (Generator + Drive Browser)

Este notebook sobe um servidor FastAPI em background dentro do Colab e o expõe via **ngrok**:

| Prefixo | Descrição |
|---------|----------|
| `/` | **Generator API** — Geração, refinamento iterativo, batch |
| `/drive` | **Drive Browser API** — Lista e serve imagens geradas do Drive |

### Pré-requisitos
- `GOOGLE_API_KEY` configurada em Secrets (ícone 🔑)
- `NGROK_AUTH_TOKEN` configurado em Secrets (obtenha em [ngrok.com](https://ngrok.com))
- Google Drive montado com pasta `clip-flow/` configurada

### Como usar
1. Execute células 1–10 para configurar
2. Execute célula 11 para subir a API e obter a URL pública
3. Acesse `{url}/docs` para Generator e `{url}/drive/docs` para Drive Browser

## ⚙️ Célula 1 — Instalar Dependências

In [ ]:
!pip install -q google-genai pillow tqdm fastapi "uvicorn[standard]" pyngrok pyyaml

import os, io, json, random, textwrap, threading, time, uuid, shutil
from pathlib import Path
from io import BytesIO
from datetime import datetime

import PIL.Image

try:
    from google.colab import userdata, drive
    IS_COLAB = True
    print('✅ Ambiente: Google Colab')
except ImportError:
    IS_COLAB = False
    print('ℹ️ Ambiente: Jupyter local')

print('✅ Dependências instaladas')

## 🔑 Célula 2 — Setup: Drive + API Keys

Configure os Secrets no Colab (ícone 🔑 no painel esquerdo):
- `GOOGLE_API_KEY` — chave do Google AI Studio
- `NGROK_AUTH_TOKEN` — token gratuito em [ngrok.com/signup](https://ngrok.com/signup)

In [ ]:
# Montar Google Drive
if IS_COLAB:
    drive.mount('/content/drive')
    print('✅ Google Drive montado')

# Carregar API Keys dos Secrets
if IS_COLAB:
    try:
        GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
        print('✅ GOOGLE_API_KEY carregada dos Secrets')
    except Exception:
        GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')
        print('⚠️ GOOGLE_API_KEY não encontrada nos Secrets — usando variável de ambiente')

    try:
        NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
        print('✅ NGROK_AUTH_TOKEN carregado dos Secrets')
    except Exception:
        NGROK_AUTH_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', '')
        print('⚠️ NGROK_AUTH_TOKEN não encontrado — configure em Secrets para URLs públicas')
else:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')
    NGROK_AUTH_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', '')

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

print(f'\n🔑 GOOGLE_API_KEY : {"✓ configurada" if GOOGLE_API_KEY else "✗ ausente"}')
print(f'🔑 NGROK_AUTH_TOKEN: {"✓ configurado" if NGROK_AUTH_TOKEN else "✗ ausente (necessário para URLs públicas)"}')

## 📁 Célula 3 — Paths e Configuração YAML

In [ ]:
import yaml

# ============================================================
# PATHS — ajuste se sua estrutura no Drive for diferente
# ============================================================
DRIVE_BASE   = Path('/content/drive/MyDrive/clip-flow')
DRIVE_REFS   = DRIVE_BASE / 'assets/backgrounds/mago'
DRIVE_OUTPUT = DRIVE_BASE / 'output/mago_gerado'
CONFIG_DIR   = DRIVE_BASE / 'config'

# Paths locais temporários no Colab (resetam com a sessão)
LOCAL_OUTPUT = Path('/content/generated_mago')
LOCAL_REFS   = Path('/content/assets/backgrounds/mago')

# Criar diretórios necessários
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
if DRIVE_OUTPUT.exists() or DRIVE_BASE.exists():
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
    CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    print(f'✅ Diretórios Drive criados/verificados')

# Saída primária: Drive (se disponível), senão local
OUTPUT_DIR = DRIVE_OUTPUT if DRIVE_OUTPUT.exists() else LOCAL_OUTPUT
print(f'📂 Saída de imagens: {OUTPUT_DIR}')

# Parâmetros padrão de lote (sobrescritos pelo batch.yaml se existir)
BATCH_DEFAULTS = {
    'n_referencias': 5,
    'pausa_segundos': 15,
    'temperatura': 0.85,
}

def load_themes_config() -> list:
    """Carrega themes.yaml do Drive. Retorna lista vazia se não existir."""
    path = CONFIG_DIR / 'themes.yaml'
    if path.exists():
        data = yaml.safe_load(path.read_text(encoding='utf-8'))
        return data if isinstance(data, list) else []
    return []

def load_batch_config() -> dict:
    """Carrega batch.yaml do Drive. Retorna BATCH_DEFAULTS se não existir."""
    path = CONFIG_DIR / 'batch.yaml'
    if path.exists():
        data = yaml.safe_load(path.read_text(encoding='utf-8'))
        return {**BATCH_DEFAULTS, **(data or {})}
    return BATCH_DEFAULTS.copy()

def save_themes_config(themes: list):
    """Salva lista de temas no Drive como YAML."""
    (CONFIG_DIR / 'themes.yaml').write_text(
        yaml.dump(themes, allow_unicode=True, default_flow_style=False),
        encoding='utf-8'
    )

print(f'✅ Configuração inicializada')
print(f'   CONFIG_DIR: {CONFIG_DIR}')
existing_themes = load_themes_config()
print(f'   themes.yaml: {len(existing_themes)} temas carregados' if existing_themes else '   themes.yaml: não existe ainda (executar Célula 10)')

## 🧬 Célula 4 — DNA do Personagem e Situações

Padrão visual estrito: **Gandalf o Cinzento** — semi-realista, cel-shading suave.

In [ ]:
# ============================================================
# DNA DO PERSONAGEM — traços imutáveis em TODOS os prompts
# Padrão: Gandalf o Cinzento (photorealistic cinematic portrait)
# ============================================================

CHARACTER_DNA = textwrap.dedent("""
    Photorealistic fantasy portrait of an ancient wise wizard with the following EXACT traits (NEVER change):

    FACE & SKIN:
    - Approximately 90 years old, deeply wrinkled weathered skin with natural skin pores
    - Prominent aquiline nose, thick bushy silver-grey eyebrows
    - Piercing intense pale blue-grey eyes with depth, wisdom, and subtle moisture reflection
    - Weathered smile lines, age spots on temples, hyper realistic skin texture

    BEARD & HAIR:
    - Very long flowing silver-white beard reaching chest, slightly unkempt natural texture
    - Individual hair strands visible, natural grey-white gradients (NEVER short or neat)
    - Wispy silver hair visible under hat brim

    HAT:
    - Wide-brimmed tall pointed grey felt hat, aged and slightly bent at tip
    - Weathered fabric texture, subtle dust and wear marks (NEVER remove, NEVER clean/new looking)

    ROBES & CLOTHING:
    - Weathered charcoal-grey wool robes with natural fabric folds and weight
    - Dark midnight blue outer layer (#1A1A3E) with worn leather belt and aged brass buckle
    - Subtle silver-gold thread embroidery on edges, slightly frayed
    - Worn brown leather boots with creases and dust

    STAFF:
    - Massive gnarled wooden staff of dark twisted oak (#3E2723)
    - Ancient runes barely visible carved into wood grain
    - Faint warm ember-golden glow at the top (#FFD54F), like dying embers

    PHYSIQUE:
    - Tall, lean, slightly hunched posture conveying age and wisdom
    - Large weathered hands with visible veins and knuckles

    RENDERING STYLE:
    - Photorealistic cinematic portrait, 85mm lens f/1.8 equivalent
    - Natural subsurface skin scattering, cinematic color grading
    - Subtle film grain, studio-quality lighting
    - Hyper realistic fabric textures, 8K detail level
    - NO cartoon, NO cel-shading, NO flat colors, NO stylization

    COLOR PALETTE (strict reference):
    Beard/Hair: #E8E8F0 (silver-white) | Hat: #4A5568 (weathered grey) |
    Robes: #1A1A3E (dark midnight blue) | Gold details: #C8A84E (aged brass) |
    Staff: #3E2723 (dark oak) | Staff glow: #FFD54F (warm ember) | Eyes: #8FA8C8 (pale blue-grey)
""").strip()

COMPOSITION = textwrap.dedent("""
    Vertical 4:5 aspect ratio (1080x1350 pixels).
    Character positioned in lower two-thirds of frame — lower third preferred.
    Upper 35-40% of image open and clear for text overlay.
    Shallow depth of field on background, f/1.8 bokeh effect.
    Soft dramatic side lighting, cinematic color grading.
    Dark atmospheric fantasy setting with warm golden accent rim lighting.
    Camera angle: slight low angle, eye-level to mid-chest framing.
""").strip()

NEGATIVE_TRAITS = (
    'NOT cartoon, NOT cel-shading, NOT flat colors, NOT anime/manga, NOT chibi, '
    'NOT stylized, NOT illustration, NOT watercolor, NOT oil painting brush strokes visible, '
    'NOT young wizard, NOT clean/new clothing, NOT bright saturated colors, '
    'NOT centered in frame, NOT bright white background, NOT without hat, NOT short beard, '
    'NOT threatening expression, NOT different colored robes, '
    'photorealism only, no stylization whatsoever'
)

# ============================================================
# 13 SITUAÇÕES PRÉ-DEFINIDAS
# ============================================================

SITUACOES = {
    'sabedoria': {
        'label': '🧠 Sabedoria / Conselho',
        'acao': 'gentle knowing smile, one eyebrow raised wisely, leaning slightly on staff, free hand gesturing as if explaining something obvious',
        'cenario': 'misty medieval forest with soft golden light filtering through ancient trees, dark moody atmosphere',
    },
    'confusao': {
        'label': '😕 Confuso / Perplexo',
        'acao': 'confused bewildered expression, head tilted to one side, one hand scratching long white beard, squinting eyes, floating question marks',
        'cenario': 'dark medieval wizard study with floating books and magical scrolls, warm candlelight',
    },
    'segunda_feira': {
        'label': '😴 Segunda-feira / Cansado',
        'acao': 'extremely tired expression, heavy eyelids, hat drooping forward, holding enormous steaming coffee goblet with both hands, dark circles under eyes',
        'cenario': 'early morning misty medieval kitchen, dawn light through stone window, cold blue atmosphere',
    },
    'vitoria': {
        'label': '🏆 Vitória / Celebração',
        'acao': 'triumphant joyful expression, staff raised high with golden magical particles erupting, hat slightly flying off, robes billowing dramatically, arms wide open',
        'cenario': 'epic medieval castle courtyard at sunset, dramatic golden light, magical sparkles everywhere',
    },
    'tecnologia': {
        'label': '📱 Tecnologia / WiFi',
        'acao': 'surprised amused expression, holding glowing crystal ball like a smartphone, hat askew from leaning forward, floating digital glowing runes around device',
        'cenario': 'anachronistic medieval office desk with candles and ancient books, blue glow from magical screen',
    },
    'cafe': {
        'label': '☕ Café / Manhã',
        'acao': 'content pleased expression, holding ornate ceramic mug with both hands, eyes slightly closed in pleasure, golden steam rising with magical sparkles',
        'cenario': 'cozy medieval kitchen corner, warm fireplace glow, morning light',
    },
    'comida': {
        'label': '🍲 Comida / Culinária',
        'acao': 'focused proud expression, stirring large bubbling cauldron with wooden ladle, small apron over robes, floating magical ingredients, colorful vapors',
        'cenario': 'magical medieval stone kitchen, warm firelight from below cauldron, herbs and potions on shelves',
    },
    'trabalho': {
        'label': '💼 Trabalho / Escritório',
        'acao': 'concentrated serious expression, sitting at wooden desk writing on parchment scroll, multiple scrolls and books open around, quill pen in hand',
        'cenario': 'dark medieval scriptorium, warm golden candlelight on desk, bookshelves reaching ceiling',
    },
    'relaxando': {
        'label': '😌 Relaxando / Feriado',
        'acao': 'blissfully asleep expression, sitting in comfortable wooden chair, hat tipped over eyes, hands folded on belly, staff leaning against wall, soft snoring aura',
        'cenario': 'cozy medieval tavern corner, warm fireplace, afternoon golden light, peaceful atmosphere',
    },
    'meditando': {
        'label': '🧘 Meditando / Zen',
        'acao': 'eyes closed, serene peaceful smile, sitting cross-legged on floating magical stone, staff hovering beside him, subtle ethereal blue-gold aura radiating outward',
        'cenario': 'mystical mountain peak at twilight, northern lights aurora in purple-blue sky, floating clouds below',
    },
    'relacionamento': {
        'label': '💕 Relacionamento / Romance',
        'acao': 'gentle warm knowing smile, holding glowing pink heart-shaped crystal in both hands, eyes twinkling with warmth, soft magical pink and blue aura',
        'cenario': 'moonlit enchanted garden with glowing flowers and fireflies, romantic soft atmosphere',
    },
    'confronto': {
        'label': '🚫 Confronto / Você Não Passa',
        'acao': 'stern but comically determined expression, staff raised high with intense golden magical energy, one hand extended forward in firm STOP gesture, dramatic magical wind blowing robes',
        'cenario': 'narrow ancient stone bridge over deep misty abyss, dramatic stormy sky, lightning in background',
    },
    'surpresa': {
        'label': '😲 Surpreso / Espantado',
        'acao': 'wide eyes of genuine shock, mouth open in surprise, both hands raised, hat blown slightly back by surprise, staff dropped sideways, exclamation sparks',
        'cenario': 'dark medieval hall with dramatic revelation lighting, magical smoke and mirrors',
    },
}

print('✅ CHARACTER_DNA, COMPOSITION, SITUACOES carregados')
print(f'   {len(SITUACOES)} situações pré-definidas + custom')

## Celula 5 — Funcoes Core: Geracao + Refinamento Nano Banana

Pipeline completo:
1. **gerar_imagem()** — geracao direta com referencias visuais
2. **refinar_imagem()** — refinamento img2img (temperatura baixa, fidelidade alta)
3. **gerar_com_refinamento()** — pipeline Nano Banana: gerar + N passes de refinamento

In [ ]:
from google import genai
from google.genai import types

# ============================================================
# CONFIGURACAO DOS MODELOS — Nano Banana Pipeline
# ============================================================
# Nano Banana      = gemini-2.5-flash-image           (estavel, producao)
# Nano Banana Exp  = gemini-2.0-flash-exp              (rapido, experimental)
# Nano Banana 3.1  = gemini-3.1-flash-image-preview    (preview)
# Nano Banana Pro  = gemini-3-pro-image-preview         (maior qualidade)

MODELOS_IMAGEM = [
    'gemini-2.5-flash-image',
    'gemini-2.0-flash-exp-image-generation',
    'gemini-3.1-flash-image-preview',
    'gemini-3-pro-image-preview',
]

TEMPERATURA        = 0.85
N_REFERENCIAS      = 5
RESOLUCAO_MAX_REF  = 1024
MAX_TENTATIVAS_429 = 2
ESPERA_BASE_429    = 60

REFERENCIAS: dict[str, PIL.Image.Image] = {}

if not GOOGLE_API_KEY:
    print('GOOGLE_API_KEY nao configurada! Volte a Celula 2.')
else:
    cliente = genai.Client(api_key=GOOGLE_API_KEY)
    print(f'Cliente Gemini configurado')
    print(f'   Modelos Nano Banana (ordem de tentativa):')
    for m in MODELOS_IMAGEM:
        print(f'     - {m}')

# ============================================================
# FUNCOES UTILITARIAS
# ============================================================

def redimensionar_para_referencia(img: PIL.Image.Image, max_size: int = 1024) -> PIL.Image.Image:
    w, h = img.size
    if max(w, h) <= max_size:
        return img
    ratio = max_size / max(w, h)
    return img.resize((int(w * ratio), int(h * ratio)), PIL.Image.LANCZOS)


def selecionar_melhores_referencias(
    referencias: dict[str, PIL.Image.Image],
    n: int = 5,
    prioridade: list[str] | None = None
) -> list[PIL.Image.Image]:
    ORDEM_PADRAO = [
        'grey_wizard_front_pose_1', 'grey_wizard_portrait_1',
        'grey_wizard_staff_raised_1', 'grey_wizard_spell_gesture_1',
        'mago_mestre_meditando_1', 'mago_mestre_tomando_cafe_1',
        'mago_mestre_lendo_grimorio_1', 'mago_mestre_invocando_criatura_1',
        'mago_mestre_cozinhando_pocao_1', 'grey_wizard_casting_magic_1',
    ]
    ordem = (prioridade or []) + [x for x in ORDEM_PADRAO if x not in (prioridade or [])]
    selecionadas = []
    for nome_base in ordem:
        if len(selecionadas) >= n:
            break
        for nome_arquivo, img in referencias.items():
            stem = Path(nome_arquivo).stem
            if nome_base in stem and img not in selecionadas:
                selecionadas.append(img)
                break
    restantes = [img for nome, img in referencias.items() if img not in selecionadas]
    random.shuffle(restantes)
    selecionadas.extend(restantes[:n - len(selecionadas)])
    return selecionadas[:n]


def pil_para_part(img: PIL.Image.Image) -> types.Part:
    img_r = redimensionar_para_referencia(img, RESOLUCAO_MAX_REF)
    buf = BytesIO()
    img_r.save(buf, format='JPEG', quality=90)
    return types.Part.from_bytes(data=buf.getvalue(), mime_type='image/jpeg')


def construir_prompt_completo(
    situacao_key: str,
    descricao_custom: str = '',
    cenario_custom: str = '',
) -> str:
    if situacao_key == 'custom':
        acao = descricao_custom or 'standing pose holding staff, wise expression'
        cenario = cenario_custom or 'dark moody medieval forest with golden atmospheric lighting'
    else:
        sit = SITUACOES.get(situacao_key, SITUACOES['sabedoria'])
        acao = sit['acao']
        cenario = sit['cenario']

    return (
        f"Generate a PHOTOREALISTIC cinematic portrait of this wizard character matching the reference images EXACTLY.\n\n"
        f"CHARACTER (replicate precisely from reference):\n{CHARACTER_DNA}\n\n"
        f"ACTION/POSE:\n{acao}\n\n"
        f"SETTING/BACKGROUND:\n{cenario}\n\n"
        f"COMPOSITION:\n{COMPOSITION}\n\n"
        f"IMPORTANT — DO NOT:\n{NEGATIVE_TRAITS}\n\n"
        f"RENDERING MANDATE: This must look like a photograph or VFX still, NOT a painting or illustration.\n"
        f"Natural skin pores, wrinkles, realistic fabric folds, cinematic shadows, shallow DOF bokeh.\n\n"
        f"The character must look IDENTICAL to the reference images in features, colors, and proportions.\n"
        f"Only the pose, expression, action, and background should change."
    )


def _tentar_gerar(modelo: str, partes: list, temperatura: float) -> PIL.Image.Image | None:
    response = cliente.models.generate_content(
        model=modelo,
        contents=partes,
        config=types.GenerateContentConfig(
            response_modalities=['IMAGE', 'TEXT'],
            temperature=temperatura,
            image_config=types.ImageConfig(aspect_ratio='4:5'),
        ),
    )
    for part in response.candidates[0].content.parts:
        if hasattr(part, 'inline_data') and part.inline_data:
            return PIL.Image.open(BytesIO(part.inline_data.data))
    return None


def gerar_imagem(
    situacao_key: str = 'sabedoria',
    descricao_custom: str = '',
    cenario_custom: str = '',
    n_referencias: int = N_REFERENCIAS,
    referencias_manual: list[PIL.Image.Image] | None = None,
    exibir: bool = False,
    salvar: bool = True,
    nome_arquivo: str = '',
) -> PIL.Image.Image | None:
    global REFERENCIAS

    if not GOOGLE_API_KEY:
        print('API Key nao configurada')
        return None

    if not REFERENCIAS and not referencias_manual:
        print('Nenhuma imagem de referencia carregada. Execute a Celula 6.')
        return None

    refs = (
        referencias_manual[:n_referencias]
        if referencias_manual
        else selecionar_melhores_referencias(REFERENCIAS, n=n_referencias)
    )

    prompt_texto = construir_prompt_completo(situacao_key, descricao_custom, cenario_custom)
    partes = []
    for img in refs:
        partes.append(pil_para_part(img))
    partes.append(
        'These are reference images of the character. '
        'Replicate the character visual style EXACTLY in your generation.'
    )
    partes.append(prompt_texto)

    label = SITUACOES.get(situacao_key, {}).get('label', situacao_key)
    print(f'Gerando: {label} | refs={len(refs)}')

    for modelo in MODELOS_IMAGEM:
        for tentativa in range(MAX_TENTATIVAS_429):
            try:
                imagem = _tentar_gerar(modelo, partes, TEMPERATURA)
                if imagem is None:
                    print(f'   {modelo}: sem imagem na resposta')
                    break

                print(f'   {modelo} -> {imagem.size[0]}x{imagem.size[1]}')

                if salvar:
                    if not nome_arquivo:
                        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
                        nome_arquivo = f'mago_{situacao_key}_{ts}'
                    caminho = OUTPUT_DIR / f'{nome_arquivo}.png'
                    imagem.save(caminho, 'PNG')
                    print(f'   Salvo: {caminho}')

                return imagem

            except Exception as e:
                msg = str(e)
                if '404' in msg or 'NOT_FOUND' in msg:
                    print(f'   {modelo}: nao disponivel (404)')
                    break
                elif '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                    espera = ESPERA_BASE_429 * (2 ** tentativa)
                    if tentativa < MAX_TENTATIVAS_429 - 1:
                        print(f'   {modelo}: 429, aguardando {espera}s...')
                        time.sleep(espera)
                    else:
                        print(f'   {modelo}: cota esgotada, tentando proximo')
                    break
                else:
                    print(f'   {modelo}: {type(e).__name__}: {msg[:120]}')
                    break

    print('Todos os modelos falharam.')
    return None


# ============================================================
# REFINAMENTO IMG2IMG — Nano Banana Refine
# ============================================================

def refinar_imagem(
    imagem_base: PIL.Image.Image,
    instrucao: str = '',
    referencias_adicionais: int = 3,
    salvar: bool = True,
    nome_arquivo: str = '',
) -> PIL.Image.Image | None:
    global REFERENCIAS

    if not GOOGLE_API_KEY:
        print('API Key nao configurada')
        return None

    partes = []
    partes.append('THIS IS THE IMAGE TO REFINE. Keep the exact same composition and character:')
    partes.append(pil_para_part(imagem_base))

    if REFERENCIAS and referencias_adicionais > 0:
        refs = selecionar_melhores_referencias(REFERENCIAS, n=referencias_adicionais)
        partes.append(
            f'These {len(refs)} images are the ORIGINAL CHARACTER REFERENCES for consistency:'
        )
        for r in refs:
            partes.append(pil_para_part(r))

    instrucao_final = instrucao or (
        'Refine this image maintaining the EXACT same composition, pose, and scene. '
        'Improve character consistency with the reference images. '
        'Ensure all character DNA traits are precisely correct.'
    )

    prompt = (
        f"{instrucao_final}\n\n"
        f"CHARACTER TRAITS TO ALWAYS MAINTAIN:\n{CHARACTER_DNA}\n\n"
        f"COMPOSITION TO MAINTAIN:\n{COMPOSITION}\n\n"
        f"DO NOT CHANGE:\n"
        f"- The overall composition and pose\n"
        f"- The background scene and mood\n"
        f"- The character action and expression\n\n"
        f"IMPROVE:\n"
        f"- Character consistency with reference images\n"
        f"- Color accuracy (strict palette: hat #4A5568, robes #1A1A3E, staff glow #FFD54F)\n"
        f"- Beard length and flow (must reach chest, white/silver, wavy)\n"
        f"- Hat shape (tall, pointed, bent tip, NEVER removed)\n"
        f"- Detail sharpness and clean cel-shading lines\n"
        f"- Gold embroidery on robes visibility"
    )

    partes.append(prompt)

    print(f'Refinando imagem... (refs adicionais: {referencias_adicionais})')

    for modelo in MODELOS_IMAGEM:
        for tentativa in range(MAX_TENTATIVAS_429):
            try:
                response = cliente.models.generate_content(
                    model=modelo,
                    contents=partes,
                    config=types.GenerateContentConfig(
                        response_modalities=['IMAGE', 'TEXT'],
                        temperature=0.4,
                        image_config=types.ImageConfig(aspect_ratio='4:5'),
                    ),
                )

                imagem_refinada = None
                for part in response.candidates[0].content.parts:
                    if hasattr(part, 'inline_data') and part.inline_data:
                        imagem_refinada = PIL.Image.open(BytesIO(part.inline_data.data))
                        break

                if imagem_refinada is None:
                    print(f'   {modelo}: sem imagem na resposta')
                    break

                print(f'   {modelo} -> refinado {imagem_refinada.size[0]}x{imagem_refinada.size[1]}')

                if salvar:
                    if not nome_arquivo:
                        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
                        nome_arquivo = f'mago_refinado_{ts}'
                    caminho = OUTPUT_DIR / f'{nome_arquivo}.png'
                    imagem_refinada.save(caminho, 'PNG')
                    print(f'   Salvo: {caminho}')

                return imagem_refinada

            except Exception as e:
                msg = str(e)
                if '404' in msg or 'NOT_FOUND' in msg:
                    print(f'   {modelo}: nao disponivel (404)')
                    break
                elif '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                    espera = ESPERA_BASE_429 * (2 ** tentativa)
                    if tentativa < MAX_TENTATIVAS_429 - 1:
                        print(f'   {modelo}: 429, aguardando {espera}s...')
                        time.sleep(espera)
                    else:
                        print(f'   {modelo}: cota esgotada')
                    break
                else:
                    print(f'   {modelo}: {type(e).__name__}: {msg[:120]}')
                    break

    print('Refinamento falhou em todos os modelos.')
    return None


def gerar_com_refinamento(
    situacao_key: str = 'sabedoria',
    descricao_custom: str = '',
    cenario_custom: str = '',
    n_referencias: int = N_REFERENCIAS,
    passes_refinamento: int = 1,
    instrucao_refinamento: str = '',
    salvar: bool = True,
    nome_arquivo: str = '',
) -> PIL.Image.Image | None:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    base_nome = nome_arquivo or f'mago_{situacao_key}_{ts}'

    print(f'=== Pipeline Nano Banana: {situacao_key} ===')
    print(f'    Geracao inicial + {passes_refinamento} passe(s) de refinamento')
    img = gerar_imagem(
        situacao_key=situacao_key,
        descricao_custom=descricao_custom,
        cenario_custom=cenario_custom,
        n_referencias=n_referencias,
        salvar=salvar,
        nome_arquivo=f'{base_nome}_v0',
        exibir=False,
    )

    if img is None:
        return None

    for i in range(passes_refinamento):
        print(f'--- Refinamento {i+1}/{passes_refinamento} ---')
        time.sleep(5)

        img_ref = refinar_imagem(
            imagem_base=img,
            instrucao=instrucao_refinamento,
            referencias_adicionais=min(4, n_referencias),
            salvar=salvar,
            nome_arquivo=f'{base_nome}_v{i+1}',
        )

        if img_ref is not None:
            img = img_ref
        else:
            print(f'   Refinamento {i+1} falhou, mantendo versao anterior')
            break

    if salvar:
        caminho_final = OUTPUT_DIR / f'{base_nome}_final.png'
        img.save(caminho_final, 'PNG')
        print(f'   FINAL: {caminho_final}')

    return img


print('Funcoes de geracao e refinamento definidas')
print('   gerar_imagem()           - geracao direta')
print('   refinar_imagem()         - refinamento img2img')
print('   gerar_com_refinamento()  - pipeline completo Nano Banana')


## 📂 Célula 6 — Carregar Referências do Drive

In [ ]:
def carregar_referencias_drive() -> dict[str, PIL.Image.Image]:
    """Carrega imagens de referência da pasta do Drive (ou local como fallback)."""
    refs = {}
    extensoes = ['.png', '.jpg', '.jpeg', '.webp']

    # Ordem de preferência: Drive > Local
    for base_dir in [DRIVE_REFS, LOCAL_REFS]:
        if not base_dir.exists():
            continue
        for ext in extensoes:
            for f in sorted(base_dir.glob(f'*{ext}')):
                try:
                    refs[f.name] = PIL.Image.open(f).convert('RGB')
                except Exception as e:
                    print(f'  ⚠️ {f.name}: {e}')
        if refs:
            print(f'📂 Referências carregadas de: {base_dir}')
            break

    return refs


REFERENCIAS = carregar_referencias_drive()

if REFERENCIAS:
    print(f'✅ {len(REFERENCIAS)} imagens de referência carregadas')
    for nome in sorted(REFERENCIAS.keys()):
        img = REFERENCIAS[nome]
        print(f'   • {nome} ({img.size[0]}x{img.size[1]})')
else:
    print('⚠️ Nenhuma referência encontrada.')
    print(f'   Esperado em: {DRIVE_REFS}')
    print('   Adicione imagens do Mago Mestre nessa pasta e re-execute esta célula.')

## 🔄 Célula 7 — Job Store + Batch Worker

Jobs ficam em memória durante a sessão do Colab. O worker processa em thread separada para não bloquear a API.

In [ ]:
JOBS: dict[str, dict] = {}


def criar_job(job_id: str):
    JOBS[job_id] = {
        'job_id': job_id,
        'status': 'queued',
        'done': 0,
        'failed': 0,
        'total': 0,
        'results': [],
        'created_at': datetime.now().isoformat(),
        'finished_at': None,
        'auto_refine': False,
        'refinement_passes': 0,
    }
    return JOBS[job_id]


def run_batch_job(job_id: str, lote: list, n_refs: int, pausa: int,
                  auto_refine: bool = False, refinement_passes: int = 1):
    job = JOBS[job_id]
    job['status'] = 'running'
    job['auto_refine'] = auto_refine
    job['refinement_passes'] = refinement_passes

    total = 0
    for item in lote:
        total += item.get('count', 1) if isinstance(item, dict) else 1
    job['total'] = total

    gerado = 0
    for item in lote:
        if isinstance(item, str):
            key, acao, cenario, count = item, '', '', 1
        elif isinstance(item, dict):
            key     = item.get('key', 'custom')
            acao    = item.get('acao', '')
            cenario = item.get('cenario', '')
            count   = item.get('count', 1)
        else:
            job['failed'] += 1
            continue

        for i in range(count):
            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            nome = f'api_{key}_{ts}'

            if auto_refine:
                img = gerar_com_refinamento(
                    situacao_key=key,
                    descricao_custom=acao,
                    cenario_custom=cenario,
                    n_referencias=n_refs,
                    passes_refinamento=refinement_passes,
                    salvar=True,
                    nome_arquivo=nome,
                )
                final_file = f'{nome}_final.png'
            else:
                img = gerar_imagem(
                    situacao_key=key,
                    descricao_custom=acao,
                    cenario_custom=cenario,
                    n_referencias=n_refs,
                    exibir=False,
                    salvar=True,
                    nome_arquivo=nome,
                )
                final_file = f'{nome}.png'

            gerado += 1
            if img:
                job['done'] += 1
                job['results'].append({
                    'theme': key,
                    'file': final_file,
                    'path': str(OUTPUT_DIR / final_file),
                    'refined': auto_refine,
                })
            else:
                job['failed'] += 1

            if gerado < total:
                time.sleep(pausa)

    job['status'] = 'completed'
    job['finished_at'] = datetime.now().isoformat()
    print(f'Job {job_id} concluido: {job["done"]} OK / {job["failed"]} falhas')


print('Job store e batch worker definidos (com suporte a auto_refine)')

## Celula 8 — Generator FastAPI v2 (porta 8000)

Pipeline Nano Banana integrado:

| Endpoint | Metodo | Descricao |
|----------|--------|----------|
| `/generate/single` | POST | Gera 1 imagem (com ou sem refinamento) |
| `/generate/refine` | POST | Refina imagem existente (N passes) |
| `/jobs/batch` | POST | Lote com lista de temas |
| `/jobs/batch/from-config` | POST | Lote usando themes.yaml |
| `/jobs/{job_id}` | GET | Status do job |
| `/jobs` | GET | Todos os jobs |
| `/themes` | GET/POST | Lista ou adiciona temas |
| `/themes/{key}` | DELETE | Remove tema |
| `/status` | GET | Estado do servico |

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Union

generator_app = FastAPI(
    title='Mago Generator API — Nano Banana Pipeline',
    description=(
        'Geracao automatica em lote com refinamento iterativo.\n\n'
        '**Pipeline Nano Banana:** gerar imagem com referencias visuais + refinar N vezes '
        'com temperatura baixa para maxima consistencia do personagem Mago Mestre.'
    ),
    version='2.0.0',
)
generator_app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


class BatchRequest(BaseModel):
    themes: list[Union[str, dict]] = Field(
        ..., description='Lista de temas: str (key) ou dict com key/acao/cenario/count'
    )
    n_refs: int = Field(default=5, ge=1, le=14, description='Referencias por geracao')
    pausa: int = Field(default=15, ge=5, description='Segundos entre geracoes')
    auto_refine: bool = Field(default=False, description='Ativar refinamento Nano Banana')
    refinement_passes: int = Field(default=1, ge=1, le=3, description='Passes de refinamento')


class SingleRequest(BaseModel):
    theme_key: str = Field(default='sabedoria', description='Chave da situacao')
    acao_custom: str = Field(default='', description='Acao customizada (se theme_key=custom)')
    cenario_custom: str = Field(default='', description='Cenario customizado (se theme_key=custom)')
    auto_refine: bool = Field(default=False, description='Ativar refinamento Nano Banana')
    refinement_passes: int = Field(default=1, ge=1, le=3, description='Passes de refinamento')


class RefineRequest(BaseModel):
    filename: str = Field(..., description='Nome do arquivo PNG a refinar (em OUTPUT_DIR)')
    instrucao: str = Field(default='', description='Instrucao especifica de refinamento (ingles)')
    referencias_adicionais: int = Field(default=3, ge=1, le=7)
    passes: int = Field(default=1, ge=1, le=3, description='Passes de refinamento iterativo')


class ThemeItem(BaseModel):
    key: str
    label: str = ''
    acao: str
    cenario: str
    count: int = 1


@generator_app.post('/generate/single', summary='Gera uma imagem individual',
    tags=['Geracao'])
def generate_single(req: SingleRequest):
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    nome = f'single_{req.theme_key}_{ts}'

    if req.auto_refine:
        img = gerar_com_refinamento(
            situacao_key=req.theme_key,
            descricao_custom=req.acao_custom,
            cenario_custom=req.cenario_custom,
            passes_refinamento=req.refinement_passes,
            salvar=True,
            nome_arquivo=nome,
        )
        final_file = f'{nome}_final.png' if img else None
    else:
        img = gerar_imagem(
            situacao_key=req.theme_key,
            descricao_custom=req.acao_custom,
            cenario_custom=req.cenario_custom,
            exibir=False,
            salvar=True,
            nome_arquivo=nome,
        )
        final_file = f'{nome}.png' if img else None

    return {
        'success': img is not None,
        'theme': req.theme_key,
        'file': final_file,
        'path': str(OUTPUT_DIR / final_file) if final_file else None,
        'refined': req.auto_refine,
        'refinement_passes': req.refinement_passes if req.auto_refine else 0,
    }


@generator_app.post('/generate/refine', summary='Refina uma imagem existente',
    tags=['Refinamento'])
def refine_existing(req: RefineRequest):
    if '/' in req.filename or '\\' in req.filename or '..' in req.filename:
        raise HTTPException(status_code=400, detail='Nome de arquivo invalido')

    path = OUTPUT_DIR / req.filename
    if not path.exists():
        raise HTTPException(status_code=404, detail=f'Imagem nao encontrada: {req.filename}')

    try:
        img_base = PIL.Image.open(path).convert('RGB')
    except Exception as e:
        raise HTTPException(status_code=400, detail=f'Erro ao abrir imagem: {e}')

    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    stem = Path(req.filename).stem

    img = img_base
    resultados = []
    for i in range(req.passes):
        nome_ref = f'{stem}_ref{i+1}_{ts}'
        img_ref = refinar_imagem(
            imagem_base=img,
            instrucao=req.instrucao,
            referencias_adicionais=req.referencias_adicionais,
            salvar=True,
            nome_arquivo=nome_ref,
        )
        if img_ref is not None:
            img = img_ref
            resultados.append({
                'pass': i + 1,
                'file': f'{nome_ref}.png',
                'path': str(OUTPUT_DIR / f'{nome_ref}.png'),
            })
        else:
            break

    return {
        'success': len(resultados) > 0,
        'original': req.filename,
        'passes_completed': len(resultados),
        'passes_requested': req.passes,
        'results': resultados,
        'final_file': resultados[-1]['file'] if resultados else None,
    }


@generator_app.post('/jobs/batch', summary='Lote com lista de temas',
    tags=['Batch'])
def create_batch(req: BatchRequest):
    job_id = str(uuid.uuid4())[:8]
    criar_job(job_id)
    threading.Thread(
        target=run_batch_job,
        args=(job_id, req.themes, req.n_refs, req.pausa),
        kwargs={'auto_refine': req.auto_refine, 'refinement_passes': req.refinement_passes},
        daemon=True,
    ).start()
    return {
        'job_id': job_id,
        'status': 'queued',
        'total_themes': len(req.themes),
        'auto_refine': req.auto_refine,
        'refinement_passes': req.refinement_passes,
    }


@generator_app.post('/jobs/batch/from-config', summary='Lote usando themes.yaml',
    tags=['Batch'])
def batch_from_config(auto_refine: bool = False, refinement_passes: int = 1):
    themes = load_themes_config()
    if not themes:
        raise HTTPException(
            status_code=404,
            detail=f'themes.yaml nao encontrado em {CONFIG_DIR}. Execute a Celula 10.'
        )
    batch_cfg = load_batch_config()
    job_id = str(uuid.uuid4())[:8]
    criar_job(job_id)
    threading.Thread(
        target=run_batch_job,
        args=(job_id, themes, batch_cfg['n_referencias'], batch_cfg['pausa_segundos']),
        kwargs={'auto_refine': auto_refine, 'refinement_passes': refinement_passes},
        daemon=True,
    ).start()
    return {
        'job_id': job_id,
        'status': 'queued',
        'total_themes': len(themes),
        'batch_config': batch_cfg,
        'auto_refine': auto_refine,
    }


@generator_app.get('/jobs/{job_id}', summary='Status de um job',
    tags=['Batch'])
def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail='Job nao encontrado')
    return job


@generator_app.get('/jobs', summary='Todos os jobs',
    tags=['Batch'])
def list_jobs():
    return {'total': len(JOBS), 'jobs': list(JOBS.values())}


@generator_app.get('/themes', summary='Lista temas disponiveis',
    tags=['Temas'])
def list_themes():
    yaml_themes = load_themes_config()
    if yaml_themes:
        return {'source': 'themes.yaml', 'themes': yaml_themes}
    return {
        'source': 'built-in',
        'themes': [
            {'key': k, 'label': v['label'], 'count': 1}
            for k, v in SITUACOES.items()
        ]
    }


@generator_app.post('/themes', summary='Adiciona tema customizado',
    tags=['Temas'])
def add_theme(theme: ThemeItem):
    themes = load_themes_config()
    themes = [t for t in themes if t.get('key') != theme.key]
    themes.append(theme.model_dump())
    save_themes_config(themes)
    return {'added': theme.key, 'total_themes': len(themes)}


@generator_app.delete('/themes/{key}', summary='Remove tema',
    tags=['Temas'])
def delete_theme(key: str):
    themes = load_themes_config()
    new_themes = [t for t in themes if t.get('key') != key]
    if len(new_themes) == len(themes):
        raise HTTPException(status_code=404, detail=f'Tema nao encontrado: {key}')
    save_themes_config(new_themes)
    return {'removed': key, 'total_themes': len(new_themes)}


@generator_app.get('/status', summary='Estado do servico',
    tags=['Sistema'])
def api_status():
    imagens_geradas = list(OUTPUT_DIR.glob('*.png'))
    return {
        'api_key_ok': bool(GOOGLE_API_KEY),
        'refs_loaded': len(REFERENCIAS),
        'drive_output_path': str(OUTPUT_DIR),
        'total_images_generated': len(imagens_geradas),
        'jobs_total': len(JOBS),
        'jobs_running': sum(1 for j in JOBS.values() if j['status'] == 'running'),
        'models': MODELOS_IMAGEM,
        'temperatura': TEMPERATURA,
        'n_referencias_default': N_REFERENCIAS,
        'pipeline': 'Nano Banana (geracao + refinamento iterativo)',
    }


print('Generator API v2 definida (porta 8000)')
print('   Endpoints:')
for route in generator_app.routes:
    if hasattr(route, 'methods'):
        for method in route.methods:
            print(f'   {method:6s} {route.path}')

## 📸 Célula 9 — Drive Browser FastAPI App (porta 8001)

Endpoints disponíveis:
- `GET /images` — lista imagens com metadados
- `GET /images/latest` — N mais recentes
- `GET /images/by-theme/{theme}` — filtra por tema
- `GET /images/{filename}` — serve a imagem PNG
- `GET /health` — estado da conexão Drive

In [ ]:
from fastapi import FastAPI, HTTPException, Query
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware

drive_app = FastAPI(
    title='Mago Drive Browser API',
    description='Lista e serve imagens geradas do Google Drive',
    version='1.0.0',
)
drive_app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


def _parse_theme_from_filename(stem: str) -> str:
    """
    Extrai tema do nome do arquivo.
    Padrões suportados:
      api_{theme}_{timestamp}  → stem.split('_')[1]
      mago_{theme}_{timestamp} → stem.split('_')[1]
      lote_{n}_{theme}_{ts}   → stem.split('_')[2]
      single_{theme}_{ts}     → stem.split('_')[1]
    """
    parts = stem.split('_')
    if len(parts) >= 3 and parts[0] in ('api', 'mago', 'single'):
        return parts[1]
    if len(parts) >= 4 and parts[0] == 'lote':
        return parts[2]
    return 'unknown'


def _list_drive_images(theme_filter: str | None = None) -> list[dict]:
    """Lista imagens PNG do OUTPUT_DIR ordenadas por data de modificação (mais recente primeiro)."""
    files = sorted(
        OUTPUT_DIR.glob('*.png'),
        key=lambda f: f.stat().st_mtime,
        reverse=True,
    )
    result = []
    for f in files:
        theme = _parse_theme_from_filename(f.stem)
        if theme_filter and theme != theme_filter:
            continue
        stat = f.stat()
        result.append({
            'filename': f.name,
            'theme': theme,
            'size_kb': round(stat.st_size / 1024, 1),
            'modified_ts': stat.st_mtime,
            'modified_at': datetime.fromtimestamp(stat.st_mtime).isoformat(),
        })
    return result


@drive_app.get('/images', summary='Lista todas as imagens geradas')
def list_images(
    theme: str | None = Query(default=None, description='Filtrar por tema (ex: sabedoria)'),
    limit: int = Query(default=20, ge=1, le=100),
    offset: int = Query(default=0, ge=0),
):
    imgs = _list_drive_images(theme)
    return {
        'total': len(imgs),
        'offset': offset,
        'limit': limit,
        'images': imgs[offset:offset + limit],
    }


@drive_app.get('/images/latest', summary='Retorna N imagens mais recentes')
def latest_images(count: int = Query(default=5, ge=1, le=50)):
    return {'count': count, 'images': _list_drive_images()[:count]}


@drive_app.get('/images/by-theme/{theme_key}', summary='Imagens filtradas por tema')
def images_by_theme(theme_key: str):
    imgs = _list_drive_images(theme_key)
    return {'theme': theme_key, 'total': len(imgs), 'images': imgs}


@drive_app.get('/images/{filename}', summary='Serve uma imagem PNG')
def get_image(filename: str):
    """Retorna a imagem PNG como streaming response (visualizável no browser)."""
    # Segurança: evitar path traversal
    if '/' in filename or '\\' in filename or '..' in filename:
        raise HTTPException(status_code=400, detail='Nome de arquivo inválido')
    path = OUTPUT_DIR / filename
    if not path.exists() or not path.is_file():
        raise HTTPException(status_code=404, detail=f'Imagem "{filename}" não encontrada')
    return FileResponse(
        path=str(path),
        media_type='image/png',
        filename=filename,
    )


@drive_app.get('/themes', summary='Temas encontrados nas imagens geradas')
def list_image_themes():
    """Retorna lista de temas únicos nas imagens geradas."""
    imgs = _list_drive_images()
    themes = sorted(set(i['theme'] for i in imgs))
    counts = {t: sum(1 for i in imgs if i['theme'] == t) for t in themes}
    return {'themes': themes, 'counts': counts}


@drive_app.get('/health', summary='Estado da conexão com o Drive')
def health():
    total_png = len(list(OUTPUT_DIR.glob('*.png'))) if OUTPUT_DIR.exists() else 0
    return {
        'drive_mounted': OUTPUT_DIR.exists(),
        'output_folder': str(OUTPUT_DIR),
        'total_images': total_png,
        'refs_folder': str(DRIVE_REFS),
        'refs_folder_exists': DRIVE_REFS.exists(),
    }


print('✅ Drive Browser API definida (porta 8001)')
print('   Endpoints:')
for route in drive_app.routes:
    if hasattr(route, 'methods'):
        for method in route.methods:
            print(f'   {method:6s} {route.path}')

## 📝 Célula 10 — Auto-gerar themes.yaml a partir de SITUACOES

Gera automaticamente o `themes.yaml` com **todos os 13 temas** do Mago Mestre + customs adicionais.
Os arquivos ficam em `clip-flow/config/` no Drive e persistem entre sessões.

In [ ]:
# ============================================================
# AUTO-GERAR themes.yaml A PARTIR DE SITUACOES
# ============================================================
# Cada situação pré-definida vira um tema no YAML com count configurável.
# Adicione customs extras na lista CUSTOMS_EXTRAS abaixo.

# Quantas imagens gerar por tema padrão (ajuste conforme cota disponível)
COUNT_POR_TEMA = 1

# Temas extras customizados (além dos 13 pré-definidos)
CUSTOMS_EXTRAS = [
    {
        'key': 'custom',
        'label': '⚔️ Você Não Passará (Bridge)',
        'acao': 'striding confidently forward on ancient stone bridge, staff raised high in defiance, dramatic expression, robes billowing in magical wind',
        'cenario': 'massive stone bridge over a fiery chasm, epic fantasy atmosphere, warm dramatic lighting from below',
        'count': 1,
    },
    {
        'key': 'custom',
        'label': '📖 Lendo Grimório',
        'acao': 'sitting in ornate wooden chair, reading a large ancient glowing book, focused expression, one hand stroking beard thoughtfully, magical runes floating from pages',
        'cenario': 'dark medieval library with towering bookshelves, warm golden candlelight, dust motes in light beams',
        'count': 1,
    },
    {
        'key': 'custom',
        'label': '🌧️ Na Chuva',
        'acao': 'walking determinedly through heavy rain, hat drooping with water, staff used as walking stick, slightly annoyed but stoic expression, wet robes clinging',
        'cenario': 'dark rainy medieval cobblestone road at night, puddles reflecting warm tavern lights in distance, moody atmosphere',
        'count': 1,
    },
]

# Configurações padrão de lote
BATCH_CONFIG = {
    'n_referencias': 5,     # imagens de referência por geração (3-7)
    'pausa_segundos': 15,   # espera entre gerações (aumentar se receber 429)
    'temperatura': 0.85,    # criatividade (0.0-2.0)
}

# ============================================================
# GERAR LISTA AUTOMATICAMENTE
# ============================================================

TEMAS_CONFIG = []

# Adicionar todos os 13 temas pré-definidos do SITUACOES
for key, sit in SITUACOES.items():
    TEMAS_CONFIG.append({
        'key': key,
        'label': sit['label'],
        'acao': sit['acao'],
        'cenario': sit['cenario'],
        'count': COUNT_POR_TEMA,
    })

# Adicionar customs extras (com key único para evitar conflito)
for i, custom in enumerate(CUSTOMS_EXTRAS):
    tema = custom.copy()
    # Gerar key único para customs
    label_slug = tema.get('label', f'custom_{i}')
    label_slug = label_slug.lower().replace(' ', '_')
    # Limpar emoji e caracteres especiais
    label_slug = ''.join(c for c in label_slug if c.isalnum() or c == '_')
    tema['key'] = f'custom_{label_slug}'[:30]
    TEMAS_CONFIG.append(tema)

# ============================================================
# SALVAR NO DRIVE
# ============================================================

if CONFIG_DIR.exists():
    save_themes_config(TEMAS_CONFIG)
    (CONFIG_DIR / 'batch.yaml').write_text(
        yaml.dump(BATCH_CONFIG, allow_unicode=True, default_flow_style=False),
        encoding='utf-8'
    )
    print(f'✅ Configuração salva no Drive:')
    print(f'   {CONFIG_DIR / "themes.yaml"} — {len(TEMAS_CONFIG)} temas')
    print(f'   {CONFIG_DIR / "batch.yaml"}')
else:
    print('⚠️ CONFIG_DIR não existe — Drive não montado?')
    print('   Execute a Célula 2 primeiro.')

print(f'\n📋 Temas configurados:')
total_imgs = sum(t.get('count', 1) for t in TEMAS_CONFIG)
for t in TEMAS_CONFIG:
    key = t.get('key', 'unknown')
    count = t.get('count', 1)
    label = t.get('label', key)
    print(f'   [{key:30s}] x{count}  {label}')
print(f'\n   Total: {total_imgs} imagens por lote')
est_min = total_imgs * (15 + BATCH_CONFIG['pausa_segundos']) // 60
est_max = total_imgs * (40 + BATCH_CONFIG['pausa_segundos']) // 60
print(f'   Estimativa: ~{est_min}-{est_max} min por lote completo')
print(f'   ⚠️  Cota gratuita: ~15 img/dia — planeje os lotes')
print(f'   💡 Com refinamento ativado, cada imagem consome 2+ chamadas')
print(f'\n   Para gerar: POST /jobs/batch/from-config')
print(f'   Com refinamento: POST /jobs/batch/from-config?refinar=true&rodadas_refinamento=1')

## Celula 11 — Iniciar API + ngrok

Sobe o servidor unico (Generator + Drive Browser montado em `/drive`) e expoe via ngrok.

- **Generator**: `{url}/docs` — geracao, refinamento, batch
- **Drive Browser**: `{url}/drive/docs` — lista e serve imagens do Drive

In [ ]:
import uvicorn
from pyngrok import ngrok, conf as ngrok_conf

API_PORT = 8000

# Montar Drive Browser API como sub-app do Generator (prefix /drive)
# Isso permite tudo rodar numa unica porta e um unico tunnel ngrok
generator_app.mount('/drive', drive_app)

# URL local (sempre funciona dentro do Colab)
LOCAL_URL = f'http://localhost:{API_PORT}'

# Header para bypass da pagina de confirmacao do ngrok free
NGROK_HEADERS = {'ngrok-skip-browser-warning': 'true'}


def _start_uvicorn(app, port: int):
    uvicorn.run(app, host='0.0.0.0', port=port, log_level='warning')


import socket

def porta_livre(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) != 0


if porta_livre(API_PORT):
    api_thread = threading.Thread(target=_start_uvicorn, args=(generator_app, API_PORT), daemon=True)
    api_thread.start()
    print(f'API server iniciado na porta {API_PORT}')
else:
    print(f'API server ja rodando na porta {API_PORT}')

time.sleep(3)

# Verificar se a API esta respondendo localmente
import requests as _req
try:
    _r = _req.get(f'{LOCAL_URL}/status', timeout=5)
    print(f'Generator API: OK ({_r.status_code})')
except Exception as _e:
    print(f'Generator API: ERRO - {_e}')

try:
    _r = _req.get(f'{LOCAL_URL}/drive/health', timeout=5)
    print(f'Drive API (/drive): OK ({_r.status_code})')
except Exception as _e:
    print(f'Drive API (/drive): ERRO - {_e}')

# Configurar ngrok para acesso externo
PUBLIC_URL = LOCAL_URL

if not NGROK_AUTH_TOKEN:
    print('\nNGROK_AUTH_TOKEN nao configurado — API acessivel apenas localmente')
    print(f'   Generator : {LOCAL_URL}/docs')
    print(f'   Drive     : {LOCAL_URL}/drive/docs')
else:
    ngrok_conf.get_default().auth_token = NGROK_AUTH_TOKEN
    try:
        for tunnel in ngrok.get_tunnels():
            ngrok.disconnect(tunnel.public_url)

        tunnel = ngrok.connect(API_PORT)
        PUBLIC_URL = tunnel.public_url

        print()
        print('=' * 60)
        print('  API ONLINE — Servidor unico com todas as rotas')
        print('=' * 60)
        print(f'  URL Publica: {PUBLIC_URL}')
        print('=' * 60)
        print(f'  Generator Docs : {PUBLIC_URL}/docs')
        print(f'  Drive Docs     : {PUBLIC_URL}/drive/docs')
        print('=' * 60)
        print()
        print('  Rotas principais:')
        print(f'    POST {PUBLIC_URL}/generate/single')
        print(f'    POST {PUBLIC_URL}/generate/refine')
        print(f'    POST {PUBLIC_URL}/jobs/batch')
        print(f'    POST {PUBLIC_URL}/jobs/batch/from-config')
        print(f'    GET  {PUBLIC_URL}/status')
        print(f'    GET  {PUBLIC_URL}/drive/images/latest')
        print(f'    GET  {PUBLIC_URL}/drive/health')
        print()
        print('  IMPORTANTE para requests externos ao ngrok free:')
        print('    Adicione header: ngrok-skip-browser-warning: true')
        print()
        print('  A URL muda a cada sessao do Colab')
        print('  Mantenha este notebook aberto para a API ficar ativa')

    except Exception as e:
        print(f'\nErro ao criar tunnel ngrok: {e}')
        print(f'   API acessivel localmente:')
        print(f'   Generator : {LOCAL_URL}/docs')
        print(f'   Drive     : {LOCAL_URL}/drive/docs')
        PUBLIC_URL = LOCAL_URL

# Aliases para compatibilidade com celula de testes
GEN_URL = PUBLIC_URL
DRV_URL = PUBLIC_URL + '/drive'
LOCAL_GEN_URL = LOCAL_URL
LOCAL_DRV_URL = LOCAL_URL + '/drive'


## Celula 12 — Testes e Exemplos

Testes usam localhost (rapido e confiavel dentro do Colab).
Para acesso externo via ngrok, use `GEN_URL`/`DRV_URL` + header `ngrok-skip-browser-warning: true`.

In [ ]:
import requests
import json

# Testes internos usam localhost (mais rapido e confiavel)
# Para acesso externo via ngrok, use GEN_URL/DRV_URL com NGROK_HEADERS
_TEST_GEN = LOCAL_GEN_URL
_TEST_DRV = LOCAL_DRV_URL

print('=' * 55)
print('  TESTE 1: Status da Generator API')
print('=' * 55)
try:
    r = requests.get(f'{_TEST_GEN}/status', timeout=10)
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f'  Erro: {e}')

print()
print('=' * 55)
print('  TESTE 2: Estado da Drive API')
print('=' * 55)
try:
    r = requests.get(f'{_TEST_DRV}/health', timeout=10)
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f'  Erro: {e}')

print()
print('=' * 55)
print('  TESTE 3: Listar temas')
print('=' * 55)
try:
    r = requests.get(f'{_TEST_GEN}/themes', timeout=10)
    data = r.json()
    print(f'  Fonte: {data["source"]}')
    for t in data['themes']:
        k = t.get('key', '?')
        c = t.get('count', 1)
        print(f'  [{k}] x{c}')
except Exception as e:
    print(f'  Erro: {e}')

print()
print('=' * 55)
print('  TESTE 4: Ultimas imagens geradas')
print('=' * 55)
try:
    r = requests.get(f'{_TEST_DRV}/images/latest', params={'count': 5}, timeout=10)
    data = r.json()
    for img in data['images']:
        print(f'  [{img["theme"]:15s}] {img["filename"]} ({img["size_kb"]} KB)')
    if not data['images']:
        print('  (nenhuma imagem gerada ainda)')
except Exception as e:
    print(f'  Erro: {e}')

print()
print('=' * 55)
print('  TESTE 5: Listar rotas disponiveis')
print('=' * 55)
try:
    r = requests.get(f'{_TEST_GEN}/openapi.json', timeout=10)
    spec = r.json()
    print(f'  API: {spec["info"]["title"]} v{spec["info"]["version"]}')
    for path, methods in spec['paths'].items():
        for method in methods:
            print(f'    {method.upper():6s} {path}')
except Exception as e:
    print(f'  Erro: {e}')

print()
print('=' * 55)
print('  EXEMPLOS DE USO (descomente para executar)')
print('=' * 55)

# Para testes internos (dentro do Colab), use _TEST_GEN / _TEST_DRV
# Para chamadas externas via ngrok, use GEN_URL / DRV_URL com NGROK_HEADERS
# Exemplo externo: requests.get(f'{GEN_URL}/status', headers=NGROK_HEADERS)

# -----------------------------------------------------------
# EXEMPLO A: Geracao simples (sem refinamento)
# -----------------------------------------------------------
# r = requests.post(f'{_TEST_GEN}/generate/single', json={
#     'theme_key': 'sabedoria',
# })
# print(r.json())

# -----------------------------------------------------------
# EXEMPLO B: Geracao com refinamento Nano Banana (1 passe)
# -----------------------------------------------------------
# r = requests.post(f'{_TEST_GEN}/generate/single', json={
#     'theme_key': 'cafe',
#     'auto_refine': True,
#     'refinement_passes': 1,
# })
# print(r.json())

# -----------------------------------------------------------
# EXEMPLO C: Geracao custom com 2 passes de refinamento
# -----------------------------------------------------------
# r = requests.post(f'{_TEST_GEN}/generate/single', json={
#     'theme_key': 'custom',
#     'acao_custom': 'riding a flying broomstick over medieval city',
#     'cenario_custom': 'night sky, full moon, rooftops below',
#     'auto_refine': True,
#     'refinement_passes': 2,
# })
# print(r.json())

# -----------------------------------------------------------
# EXEMPLO D: Refinar imagem existente
# -----------------------------------------------------------
# r = requests.post(f'{_TEST_GEN}/generate/refine', json={
#     'filename': 'single_sabedoria_20260305_120000.png',
#     'instrucao': 'Make the beard longer, ensure hat is dark blue-grey #4A5568',
#     'passes': 2,
# })
# print(r.json())

# -----------------------------------------------------------
# EXEMPLO E: Lote com refinamento automatico
# -----------------------------------------------------------
# r = requests.post(f'{_TEST_GEN}/jobs/batch', json={
#     'themes': ['sabedoria', 'cafe', 'confronto'],
#     'n_refs': 5,
#     'pausa': 15,
#     'auto_refine': True,
#     'refinement_passes': 1,
# })
# job = r.json()
# print(f'Job criado: {job["job_id"]}')
#
# import time
# while True:
#     r = requests.get(f'{_TEST_GEN}/jobs/{job["job_id"]}')
#     j = r.json()
#     print(f'  {j["status"]} | {j["done"]}/{j["total"]} | {j["failed"]} falhas')
#     if j['status'] in ('completed', 'failed'):
#         break
#     time.sleep(10)

# -----------------------------------------------------------
# EXEMPLO F: Lote do themes.yaml COM refinamento
# -----------------------------------------------------------
# r = requests.post(
#     f'{_TEST_GEN}/jobs/batch/from-config',
#     params={'auto_refine': True, 'refinement_passes': 1}
# )
# print(r.json())

print()
print('Docs completos (acessar no browser):')
print(f'   {GEN_URL}/docs  <- Generator API (geracao + refinamento)')
print(f'   {GEN_URL}/drive/docs  <- Drive Browser API (imagens)')
if GEN_URL != LOCAL_GEN_URL:
    print()
    print('Para requests externos ao ngrok, adicionar header:')
    print('   ngrok-skip-browser-warning: true')